# M2 학습예산 원인진단 — Dunnhumby seed 43

seed 42에서 관찰된 300 epoch Top-10 역전이 seed 특이성인지 확인합니다. **M1과 M2를 모두** L2 `1e-3`, 64차원, 300 epoch로 처음부터 학습하고 25 epoch마다 신규상품 개발분할을 평가합니다.

- M2: 개인 구매이력 기반 `q_N·q_V` 표현, `rho=0.05`
- K=1 균일 음성, binary graph, BPR 표본가중 없음
- 하나의 forward·optimizer에서 공동학습, 외부 재정렬 없음
- final test·holdout 없음
- 중간 최고 epoch를 선택하지 않고 100·300 epoch만 사전 비교

이 M2는 `q_C`를 쓰지 않으므로 **CLV 구성요소 기반 M2**로만 해석합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '401f71d9e10ff7cd2f1afc208e3f81aee19450f9'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip() == REVIEWED_SHA

In [ ]:
import json
import torch
import lightgcn_clv_m2_capacity_search as search

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
OUT_DIR = '/content/drive/MyDrive/논문/data/results_v3_dunnhumby_clv_m2_training_budget_seed43_v1'
cfg = search.configure_capacity_search(
    conditions=('baseline',), seeds=(43,), out_dir=OUT_DIR,
)
summary = search.preflight_summary(cfg)
specs = search.arm_specifications(cfg)
assert summary['seeds'] == [43]
assert list(summary['conditions']) == ['baseline']
assert [s['model_id'] for s in specs] == [search.M1_MODEL_ID, search.M2_MODEL_ID]
assert all(s['pref_reg'] == 1e-3 for s in specs)
assert cfg.epochs == 300 and cfg.eval_every == 25
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
curve = search.run_capacity_search(cfg)

In [ ]:
from IPython.display import display
print('1) 전체 곡선'); display(curve)
print('2) 동일 epoch M2-M1'); display(curve.attrs['gap'])
print('3) 기계 판독'); print(json.dumps(curve.attrs['reading'], ensure_ascii=False, indent=2))
print('저장:', json.dumps(curve.attrs['result_paths'], ensure_ascii=False, indent=2))

In [ ]:
import matplotlib.pyplot as plt
gap=curve.attrs['gap']
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
for model_id,line in curve.groupby('model_id'):
    axes[0].plot(line.epoch,line['recall@10'],marker='o',label=model_id.split('_')[0].upper())
axes[1].plot(gap.epoch,gap['recall@10'],marker='o',label='M2-M1')
for ax in axes:
    ax.axvline(100,color='gray',linestyle='--'); ax.set_xlabel('epoch'); ax.legend()
axes[0].set_title('Development Recall@10'); axes[1].set_title('M2-M1 Recall@10')
axes[1].axhline(0,color='black',linewidth=.8)
plt.tight_layout(); plt.show()